# Flat RoBERTa — 5-Fold Cross-Validation with Rotating Validation Fold

This notebook replaces the previous multi-seed train/validation/test experiment with a **single-seed CV design**:

- Seed: `42` only.
- The prepared dataset and its fixed five-fold assignments are loaded from `dataset_with_test_fold.xlsx`.
- For each CV round: one fold is used as **test**, the next fold is used as **validation**, and the remaining three folds are used for **training**.
- The validation fold is used only for early stopping and best-checkpoint selection.
- The final performance report is based on the five held-out test folds, i.e., out-of-fold test predictions.


In [ ]:

# =========================
# 0) Imports and Drive mount
# =========================
import os
import re
import gc
import json
import math
import time
import shutil
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)

try:
    from safetensors.torch import load_file, save_file
    HAS_SAFETENSORS = True
except Exception:
    HAS_SAFETENSORS = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive was not mounted automatically:', e)


In [ ]:

# =========================
# 1) Configuration
# =========================
MODEL_ZOO = {
    "roberta": "roberta-base",
}

# Single-seed CV: do not loop over multiple seeds.
SEED = 42
N_FOLDS = 5

BATCH_SIZES = [8]
FINETUNE_MODES = ["full"]
LOSS_VARIANTS = ["weighted_focal"]

MAX_EPOCHS = 30
MAX_LEN = 128
FOCAL_GAMMA = 1.5
MAX_CLASS_WEIGHT_CLIP = 7.5
EARLY_STOPPING_PATIENCE = 3
SAVE_TOTAL_LIMIT = 1
MAX_GRAD_NORM = 1.0

POLICY_NAME = "flat_regularized_wfocal_g15"

DROPOUT = 0.20
CLASSIFIER_DROPOUT = 0.25

# This strategy matches: one fold test, one fold validation, three folds train.
# For each round: test = fold i, validation = fold (i + 1) mod 5, train = remaining folds.
CV_VAL_STRATEGY = "rotate_fold"

# Prepared dataset with the exact zero-based test-fold assignment used by all experiments.
CV_DATASET_PATH = "data/dataset_with_test_fold.xlsx"

OUT_DIR = "results/02_flat_transformer"
LOCAL_OUT_DIR = "/content/02_flat_transformer"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_OUT_DIR, exist_ok=True)

MASTER_RESULTS_CSV_LOCAL = os.path.join(LOCAL_OUT_DIR, "MASTER_FLAT_5FOLD_CV_SEED42.csv")
MASTER_RESULTS_CSV_DRIVE = os.path.join(OUT_DIR, "MASTER_FLAT_5FOLD_CV_SEED42.csv")

print("CV_DATASET_PATH:", CV_DATASET_PATH)
print("OUT_DIR:", OUT_DIR)
print("LOCAL_OUT_DIR:", LOCAL_OUT_DIR)
print("Seed:", SEED)
print("Folds:", N_FOLDS)
print("Validation strategy:", CV_VAL_STRATEGY)


In [ ]:

# =========================
# 2) Reproducibility and I/O helpers
# =========================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything(SEED)


def save_json(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path: str):
    with open(path, "r") as f:
        return json.load(f)


def sync_file_to_drive(local_path: str, drive_path: str):
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    tmp_drive = drive_path + ".tmp"
    shutil.copy2(local_path, tmp_drive)
    os.replace(tmp_drive, drive_path)
    time.sleep(0.2)


def append_master_row(row: Dict[str, Any]):
    df_new = pd.DataFrame([row])
    if os.path.exists(MASTER_RESULTS_CSV_LOCAL):
        old = pd.read_csv(MASTER_RESULTS_CSV_LOCAL)
        out = pd.concat([old, df_new], ignore_index=True)
    else:
        out = df_new

    tmp_local = MASTER_RESULTS_CSV_LOCAL + ".tmp"
    out.to_csv(tmp_local, index=False)
    os.replace(tmp_local, MASTER_RESULTS_CSV_LOCAL)
    sync_file_to_drive(MASTER_RESULTS_CSV_LOCAL, MASTER_RESULTS_CSV_DRIVE)
    print("MASTER updated:", MASTER_RESULTS_CSV_DRIVE)


In [ ]:

# =========================
# 3) Load the prepared dataset with fixed fold assignments
# =========================
def load_prepared_cv_dataset(path: str, n_folds: int = 5) -> pd.DataFrame:
    """Load the prepared dataset unchanged and validate its saved folds."""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    elif path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported dataset format: {path}")

    required = ["id", "orig_id", "source_split", "cv_index", "text", "L1", "L2", "test_fold"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Prepared CV dataset is missing required columns: {missing}. "
            "Use full_dataset_with_test_fold.xlsx from the preparation notebook."
        )
    if df[required].isna().any().any():
        bad_columns = df[required].columns[df[required].isna().any()].tolist()
        raise ValueError(f"Prepared CV dataset contains missing values in: {bad_columns}")

    df["cv_index"] = pd.to_numeric(df["cv_index"], errors="raise").astype(int)
    df["test_fold"] = pd.to_numeric(df["test_fold"], errors="raise").astype(int)
    if df["cv_index"].duplicated().any():
        raise ValueError("cv_index is not unique in the prepared CV dataset.")
    if df["id"].astype(str).duplicated().any():
        raise ValueError("id is not unique in the prepared CV dataset.")

    expected_folds = set(range(n_folds))
    observed_folds = set(df["test_fold"].unique().tolist())
    if observed_folds != expected_folds:
        raise ValueError(
            f"Expected saved folds {sorted(expected_folds)}, found {sorted(observed_folds)}."
        )

    # Do not repeat text cleaning, label normalization, merging, or deduplication.
    return df.sort_values("cv_index").reset_index(drop=True)


def build_l2_to_l1_map(df: pd.DataFrame) -> Dict[str, str]:
    conflicts = df.groupby("L2")["L1"].nunique()
    bad = conflicts[conflicts > 1]
    if len(bad) > 0:
        examples = bad.index.tolist()[:10]
        raise ValueError(f"Some L2 labels map to more than one L1. Fix label normalization first. Examples: {examples}")
    return df.groupby("L2")["L1"].first().to_dict()


full_df = load_prepared_cv_dataset(CV_DATASET_PATH, n_folds=N_FOLDS)
l2_to_l1 = build_l2_to_l1_map(full_df)
ALL_L2_LABELS = sorted(full_df["L2"].unique().tolist())
ALL_L1_LABELS = sorted(full_df["L1"].unique().tolist())
L2_LAB2ID = {lab: i for i, lab in enumerate(ALL_L2_LABELS)}
L2_ID2LAB = {i: lab for lab, i in L2_LAB2ID.items()}

print("Full dataset:", full_df.shape)
print("L1 classes:", len(ALL_L1_LABELS), ALL_L1_LABELS)
print("L2 classes:", len(ALL_L2_LABELS))
print("Saved test-fold sizes:")
display(full_df["test_fold"].value_counts().sort_index().rename("n").to_frame())
print("Smallest L2 supports:")
display(full_df["L2"].value_counts().sort_values().head(10).rename("support").to_frame())

rare = full_df["L2"].value_counts()
rare = rare[rare < N_FOLDS]
if len(rare) > 0:
    print("WARNING: Some L2 classes have fewer samples than N_FOLDS.")
    print("These classes cannot appear in every test fold. Aggregate out-of-fold reporting is still useful, but fold-level metrics may be unstable.")
    display(rare.rename("support").to_frame())


In [ ]:

# =========================
# 4) Reconstruct CV roles from the saved test_fold column
# =========================
def make_rotating_cv_splits_from_saved_folds(
    df: pd.DataFrame,
    n_folds: int = 5,
) -> List[Dict[str, Any]]:
    """Reconstruct roles from saved folds without calling a splitter."""
    folds = [
        np.flatnonzero(df["test_fold"].to_numpy() == fold_id).astype(int)
        for fold_id in range(n_folds)
    ]
    if any(len(indices) == 0 for indices in folds):
        raise ValueError("At least one saved test fold is empty.")

    all_idx = set(range(len(df)))
    split_rows = []
    split_specs = []

    for test_fold in range(n_folds):
        val_fold = (test_fold + 1) % n_folds
        test_idx = folds[test_fold]
        val_idx = folds[val_fold]
        train_idx = np.asarray(sorted(list(all_idx - set(test_idx.tolist()) - set(val_idx.tolist()))), dtype=int)

        spec = {
            "fold": int(test_fold),
            "val_fold": int(val_fold),
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_size": int(len(train_idx)),
            "val_size": int(len(val_idx)),
            "test_size": int(len(test_idx)),
        }
        split_specs.append(spec)

        for role, idxs in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
            tmp = df.iloc[idxs].copy()
            counts = tmp.groupby(["L1", "L2"]).size().reset_index(name="n")
            counts["fold"] = test_fold
            counts["role"] = role
            split_rows.append(counts)

        train_labels = set(df.iloc[train_idx]["L2"])
        val_test_labels = set(df.iloc[val_idx]["L2"]) | set(df.iloc[test_idx]["L2"])
        missing_train = sorted(val_test_labels - train_labels)
        if missing_train:
            print(f"WARNING fold {test_fold}: these L2 labels occur in val/test but not train:", missing_train)
            print("The model has output units for them, but cannot learn them in this fold. Consider fewer folds or no separate validation fold for rare classes.")

    split_audit = pd.concat(split_rows, ignore_index=True)
    split_audit_path = os.path.join(OUT_DIR, "cv_fold_distribution_by_L1_L2.csv")
    split_audit.to_csv(split_audit_path, index=False)
    print("Saved split audit:", split_audit_path)
    return split_specs


cv_splits = make_rotating_cv_splits_from_saved_folds(full_df, n_folds=N_FOLDS)

all_test_indices = np.concatenate([s["test_idx"] for s in cv_splits])
if len(all_test_indices) != len(full_df) or len(np.unique(all_test_indices)) != len(full_df):
    raise ValueError("Saved test folds must cover every requirement exactly once.")
print("Validated: saved folds cover every requirement exactly once as test data.")

pd.DataFrame([
    {k: v for k, v in s.items() if not k.endswith("_idx")}
    for s in cv_splits
])


In [ ]:

# =========================
# 5) Dataset, label encoding, class weights
# =========================
def encode_labels(df: pd.DataFrame, col: str, lab2id: Dict[str, int]) -> List[int]:
    vals = df[col].astype(str).tolist()
    missing = sorted(set(vals) - set(lab2id.keys()))
    if missing:
        raise ValueError(f"Unknown labels in {col}: {missing[:10]}")
    return [lab2id[v] for v in vals]


class SimpleTextDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, y: List[int], tokenizer, max_len: int):
        self.df = df.reset_index(drop=True)
        self.y = np.asarray(y, dtype=np.int64)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        text = str(self.df.loc[idx, "text"])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.y[idx], dtype=torch.long)
        return item


def compute_balanced_class_weights(
    label_ids: np.ndarray,
    num_labels: int,
    normalize_mean: bool = True,
    max_weight: Optional[float] = None,
) -> torch.FloatTensor:
    label_ids = np.asarray(label_ids, dtype=int)
    counts = np.bincount(label_ids, minlength=num_labels).astype(np.float64)
    counts[counts == 0] = 1.0
    n = counts.sum()
    weights = n / (num_labels * counts)
    if normalize_mean:
        weights = weights / max(weights.mean(), 1e-12)
    if max_weight is not None:
        weights = np.minimum(weights, float(max_weight))
    return torch.tensor(weights, dtype=torch.float32)


In [ ]:

# =========================
# 6) Model freezing and hyperparameters
# =========================
def freeze_bottom_unfreeze_last_k(model, k: int):
    for p in model.parameters():
        p.requires_grad = False

    head_keywords = ["classifier", "score", "pre_classifier", "qa_outputs"]
    for n, p in model.named_parameters():
        if any(hk in n for hk in head_keywords):
            p.requires_grad = True

    base = getattr(model, model.base_model_prefix)
    layers = None
    if hasattr(base, "encoder") and hasattr(base.encoder, "layer"):
        layers = base.encoder.layer
    elif hasattr(base, "transformer") and hasattr(base.transformer, "layer"):
        layers = base.transformer.layer

    if layers is None:
        raise ValueError("Could not find transformer layers for freezing/unfreezing.")

    k = max(0, min(k, len(layers)))
    for layer in layers[-k:]:
        for p in layer.parameters():
            p.requires_grad = True
    return model


def hparams_for_mode(finetune_mode: str):
    if finetune_mode == "full":
        return {"lr": 1e-5, "wd": 0.05, "warmup_ratio": 0.08}
    if finetune_mode == "freeze0":
        return {"lr": 5e-4, "wd": 0.0, "warmup_ratio": 0.10}
    if finetune_mode == "freeze1":
        return {"lr": 6e-5, "wd": 0.01, "warmup_ratio": 0.06}
    if finetune_mode == "freeze2":
        return {"lr": 5e-5, "wd": 0.01, "warmup_ratio": 0.06}
    return {"lr": 2e-5, "wd": 0.01, "warmup_ratio": 0.06}


In [ ]:

# =========================
# 7) Focal loss and Trainer
# =========================
class MulticlassFocalLoss(nn.Module):
    def __init__(self, alpha: Optional[torch.Tensor] = None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        log_probs = F.log_softmax(logits, dim=-1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        ce = -log_pt
        loss = (1.0 - pt).pow(self.gamma) * ce
        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)
            loss = alpha[targets] * loss
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class FlatLossTrainer(Trainer):
    VALID_LOSSES = {"ce", "weighted_ce", "focal", "weighted_focal"}

    def __init__(self, loss_variant: str = "ce", class_weights: Optional[torch.Tensor] = None, gamma: float = 2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        loss_variant = str(loss_variant).lower().strip()
        if loss_variant not in self.VALID_LOSSES:
            raise ValueError(f"Unknown loss_variant={loss_variant}. Use one of {sorted(self.VALID_LOSSES)}")
        self.loss_variant = loss_variant
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        if self.loss_variant == "ce":
            loss = F.cross_entropy(logits, labels)
        elif self.loss_variant == "weighted_ce":
            if self.class_weights is None:
                raise ValueError("weighted_ce requires class_weights.")
            loss = F.cross_entropy(logits, labels, weight=self.class_weights.to(logits.device))
        elif self.loss_variant == "focal":
            loss = MulticlassFocalLoss(alpha=None, gamma=self.gamma, reduction="mean")(logits, labels)
        elif self.loss_variant == "weighted_focal":
            if self.class_weights is None:
                raise ValueError("weighted_focal requires class_weights.")
            # Class weights are used once as focal alpha_y; do not also pass them into CE.
            loss = MulticlassFocalLoss(alpha=self.class_weights.to(logits.device), gamma=self.gamma, reduction="mean")(logits, labels)

        return (loss, outputs) if return_outputs else loss


def make_compute_metrics():
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {
            "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
            "accuracy": float(accuracy_score(labels, preds)),
            "balanced_acc": float(balanced_accuracy_score(labels, preds)),
            "precision_macro": float(precision_score(labels, preds, average="macro", zero_division=0)),
            "recall_macro": float(recall_score(labels, preds, average="macro", zero_division=0)),
        }
    return compute_metrics


In [ ]:

# =========================
# 8) Optional LayerNorm checkpoint fix and plotting helpers
# =========================
def fix_layernorm_keys_in_checkpoint(checkpoint_dir: str):
    if not HAS_SAFETENSORS:
        return False
    model_path = os.path.join(checkpoint_dir, "model.safetensors")
    if not os.path.exists(model_path):
        return False
    sd = load_file(model_path)
    needs_fix = any((".LayerNorm.beta" in k) or (".LayerNorm.gamma" in k) for k in sd.keys())
    if not needs_fix:
        return False
    fixed_sd = {}
    for k, v in sd.items():
        new_k = k.replace(".LayerNorm.beta", ".LayerNorm.bias").replace(".LayerNorm.gamma", ".LayerNorm.weight")
        fixed_sd[new_k] = v
    backup_path = os.path.join(checkpoint_dir, "model_original_unfixed.safetensors")
    if not os.path.exists(backup_path):
        shutil.copy2(model_path, backup_path)
    save_file(fixed_sd, model_path)
    return True


class FixLayerNormCheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        try:
            fix_layernorm_keys_in_checkpoint(ckpt_dir)
        except Exception as e:
            print(f"Could not fix checkpoint {ckpt_dir}: {e}")
        return control


def plot_loss_curves(run_dir: str, log_path: str = "trainer_log_history.json") -> bool:
    p = os.path.join(run_dir, log_path)
    if not os.path.exists(p):
        return False
    hist = load_json(p)
    train_epochs, train_loss = [], []
    eval_epochs, eval_loss = [], []
    for r in hist:
        if "loss" in r:
            train_epochs.append(r.get("epoch", r.get("step", len(train_epochs))))
            train_loss.append(r["loss"])
        if "eval_loss" in r:
            eval_epochs.append(r.get("epoch", r.get("step", len(eval_epochs))))
            eval_loss.append(r["eval_loss"])
    if not train_loss and not eval_loss:
        return False
    plt.figure(figsize=(7, 5))
    if train_loss:
        plt.plot(train_epochs, train_loss, marker="o", label="train_loss")
    if eval_loss:
        plt.plot(eval_epochs, eval_loss, marker="o", label="val_loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Loss curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, "loss_curve.png"), dpi=200)
    plt.close()
    return True


def plot_all_loss_overlays(root_dir: str, out_png: str = "ALL_FOLD_LOSS_CURVES.png"):
    run_dirs = sorted([os.path.join(root_dir, d) for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d)) and d.startswith("flat_cv")])
    plt.figure(figsize=(10, 6))
    n = 0
    for rd in run_dirs:
        p = os.path.join(rd, "trainer_log_history.json")
        if not os.path.exists(p):
            continue
        hist = load_json(p)
        eval_epochs, eval_loss = [], []
        for r in hist:
            if "eval_loss" in r:
                eval_epochs.append(r.get("epoch", r.get("step", len(eval_epochs))))
                eval_loss.append(r["eval_loss"])
        if eval_loss:
            plt.plot(eval_epochs, eval_loss, marker="o", label=os.path.basename(rd))
            n += 1
    if n == 0:
        plt.close()
        return False
    plt.xlabel("epoch")
    plt.ylabel("validation loss")
    plt.title("Validation loss over CV folds")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize="x-small")
    plt.tight_layout()
    plt.savefig(os.path.join(root_dir, out_png), dpi=200, bbox_inches="tight")
    plt.close()
    return True


def plot_confusion(cm: np.ndarray, labels: List[str], title: str, out_path: str, annotate_threshold: int = 15):
    fig_size = max(8, min(18, 0.45 * len(labels)))
    plt.figure(figsize=(fig_size, fig_size))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    if len(labels) <= annotate_threshold:
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                if cm[i, j] != 0:
                    plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()


In [ ]:

# =========================
# 9) Hierarchical metrics, error analysis, and reports
# =========================
def flat_metrics_from_labels(y_true: List[str], y_pred: List[str], labels: Optional[List[str]] = None) -> Dict[str, float]:
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    }


def add_parent_predictions(pred_df: pd.DataFrame, l2_to_l1: Dict[str, str]) -> pd.DataFrame:
    out = pred_df.copy()
    out["true_L1"] = out["true_L2"].map(l2_to_l1)
    out["pred_L1"] = out["pred_L2"].map(l2_to_l1)
    return out


def hierarchical_path_metrics(pred_df: pd.DataFrame) -> Dict[str, float]:
    hp_vals, hr_vals, hf_vals = [], [], []
    for r in pred_df.itertuples(index=False):
        true_path = {r.true_L1, r.true_L2}
        pred_path = {r.pred_L1, r.pred_L2}
        inter = len(true_path & pred_path)
        hp = inter / max(len(pred_path), 1)
        hr = inter / max(len(true_path), 1)
        hf = 0.0 if (hp + hr) == 0 else (2 * hp * hr) / (hp + hr)
        hp_vals.append(hp)
        hr_vals.append(hr)
        hf_vals.append(hf)
    return {
        "HP": float(np.mean(hp_vals)) if hp_vals else 0.0,
        "HR": float(np.mean(hr_vals)) if hr_vals else 0.0,
        "HF1": float(np.mean(hf_vals)) if hf_vals else 0.0,
        "parent_accuracy": float(accuracy_score(pred_df["true_L1"], pred_df["pred_L1"])) if len(pred_df) else 0.0,
    }


def error_decomposition(pred_df: pd.DataFrame) -> Dict[str, Any]:
    n = len(pred_df)
    correct_l2 = (pred_df["true_L2"] == pred_df["pred_L2"])
    correct_parent = (pred_df["true_L1"] == pred_df["pred_L1"])
    wrong_l2_correct_l1 = (~correct_l2) & correct_parent
    wrong_l1 = ~correct_parent
    return {
        "n_samples": int(n),
        "Correct L2": int(correct_l2.sum()),
        "Wrong L2 but correct L1": int(wrong_l2_correct_l1.sum()),
        "Wrong L1": int(wrong_l1.sum()),
        "Correct L2 pct": float(correct_l2.mean()) if n else 0.0,
        "Wrong L2 but correct L1 pct": float(wrong_l2_correct_l1.mean()) if n else 0.0,
        "Wrong L1 pct": float(wrong_l1.mean()) if n else 0.0,
    }


def classification_report_df(y_true: List[str], y_pred: List[str], labels: List[str]) -> pd.DataFrame:
    rep = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    rows = []
    for lab in labels:
        d = rep.get(lab, {})
        rows.append({
            "label": lab,
            "precision": d.get("precision", np.nan),
            "recall": d.get("recall", np.nan),
            "f1": d.get("f1-score", np.nan),
            "support": d.get("support", 0),
        })
    out = pd.DataFrame(rows)
    zero_support = out["support"].fillna(0).eq(0)
    out.loc[zero_support, ["precision", "recall", "f1"]] = np.nan
    return out


def per_parent_within_l2_metrics(pred_df: pd.DataFrame, l2_to_l1: Dict[str, str], all_l1_labels: List[str]) -> pd.DataFrame:
    rows = []
    for parent in all_l1_labels:
        sub = pred_df[pred_df["true_L1"].eq(parent)].copy()
        children = sorted([l2 for l2, p in l2_to_l1.items() if p == parent])
        support = len(sub)
        if support == 0:
            rows.append({
                "parent": parent,
                "support": 0,
                "parent_recall_as_parent": np.nan,
                "within_parent_l2_macro_f1": np.nan,
                "within_parent_l2_weighted_f1": np.nan,
                "sibling_error_rate": np.nan,
                "cross_parent_error_rate": np.nan,
            })
            continue
        correct_l2 = sub["true_L2"].eq(sub["pred_L2"])
        correct_parent = sub["true_L1"].eq(sub["pred_L1"])
        rows.append({
            "parent": parent,
            "support": int(support),
            "parent_recall_as_parent": float(correct_parent.mean()),
            "within_parent_l2_macro_f1": float(f1_score(sub["true_L2"], sub["pred_L2"], labels=children, average="macro", zero_division=0)),
            "within_parent_l2_weighted_f1": float(f1_score(sub["true_L2"], sub["pred_L2"], labels=children, average="weighted", zero_division=0)),
            "sibling_error_rate": float(((~correct_l2) & correct_parent).mean()),
            "cross_parent_error_rate": float((~correct_parent).mean()),
        })
    return pd.DataFrame(rows)


def prefix_metrics(prefix: str, d: Dict[str, Any]) -> Dict[str, Any]:
    return {f"{prefix}_{k}": v for k, v in d.items()}


In [ ]:

# =========================
# 10) TrainingArguments builder
# =========================
def build_training_args(run_dir: str, seed: int, batch_size: int, hp: Dict[str, Any], warmup_steps: int):
    common = dict(
        output_dir=run_dir,
        seed=seed,
        data_seed=seed,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=MAX_EPOCHS,
        save_strategy="epoch",
        optim="adamw_torch",
        learning_rate=hp["lr"],
        weight_decay=hp["wd"],
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        max_grad_norm=MAX_GRAD_NORM,
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        fp16=torch.cuda.is_available(),
        full_determinism=True,
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **common)


In [ ]:

# =========================
# 11) Run one CV fold
# =========================
def run_one_cv_fold(
    full_df: pd.DataFrame,
    split_spec: Dict[str, Any],
    model_name: str,
    model_key: str,
    batch_size: int,
    finetune_mode: str,
    loss_variant: str,
    seed: int = 42,
):
    fold = int(split_spec["fold"])
    train_df = full_df.iloc[split_spec["train_idx"]].reset_index(drop=True)
    val_df   = full_df.iloc[split_spec["val_idx"]].reset_index(drop=True)
    test_df  = full_df.iloc[split_spec["test_idx"]].reset_index(drop=True)

    hp = hparams_for_mode(finetune_mode)
    seed_everything(seed)

    y_train = encode_labels(train_df, "L2", L2_LAB2ID)
    y_val   = encode_labels(val_df, "L2", L2_LAB2ID)
    y_test  = encode_labels(test_df, "L2", L2_LAB2ID)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    train_ds = SimpleTextDataset(train_df, y_train, tokenizer, MAX_LEN)
    val_ds   = SimpleTextDataset(val_df, y_val, tokenizer, MAX_LEN)
    test_ds  = SimpleTextDataset(test_df, y_test, tokenizer, MAX_LEN)

    config = AutoConfig.from_pretrained(
        model_name,
        num_labels=len(L2_LAB2ID),
        id2label=L2_ID2LAB,
        label2id=L2_LAB2ID,
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    if finetune_mode.startswith("freeze"):
        k = int(finetune_mode.replace("freeze", ""))
        model = freeze_bottom_unfreeze_last_k(model, k)

    run_id = f"flat_cv{fold}_{model_key}_{finetune_mode}_bs{batch_size}_loss{loss_variant}_seed{seed}"
    run_dir = os.path.join(OUT_DIR, run_id)
    os.makedirs(run_dir, exist_ok=True)

    for item in os.listdir(run_dir):
        path = os.path.join(run_dir, item)
        if item.startswith("checkpoint-") and os.path.isdir(path):
            shutil.rmtree(path)

    steps_per_epoch = int(math.ceil(len(train_ds) / batch_size))
    total_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(hp.get("warmup_ratio", 0.06) * total_steps)
    args = build_training_args(run_dir, seed, batch_size, hp, warmup_steps)

    class_w = None
    if loss_variant in {"weighted_ce", "weighted_focal"}:
        class_w = compute_balanced_class_weights(
            np.asarray(y_train),
            num_labels=len(L2_LAB2ID),
            normalize_mean=True,
            max_weight=MAX_CLASS_WEIGHT_CLIP,
        )

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("=" * 80)
    print(f"Fold {fold} | train={len(train_df)} val={len(val_df)} test={len(test_df)} | loss={loss_variant}")
    print(f"Trainable params: {trainable:,}/{total:,} ({100 * trainable / total:.2f}%)")

    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=make_compute_metrics(),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE), FixLayerNormCheckpointCallback()],
    )
    try:
        trainer = FlatLossTrainer(
            **trainer_kwargs,
            processing_class=tokenizer,
            loss_variant=loss_variant,
            class_weights=class_w,
            gamma=FOCAL_GAMMA,
        )
    except TypeError:
        trainer = FlatLossTrainer(
            **trainer_kwargs,
            loss_variant=loss_variant,
            class_weights=class_w,
            gamma=FOCAL_GAMMA,
        )

    trainer.train()
    save_json(trainer.state.log_history, os.path.join(run_dir, "trainer_log_history.json"))
    loss_saved = plot_loss_curves(run_dir, "trainer_log_history.json")

    val_pred = trainer.predict(val_ds)
    test_pred = trainer.predict(test_ds)

    val_y_pred = np.argmax(val_pred.predictions, axis=1)
    test_y_pred = np.argmax(test_pred.predictions, axis=1)

    val_pred_df = val_df[["id", "orig_id", "source_split", "cv_index", "text", "L1", "L2"]].copy()
    val_pred_df = val_pred_df.rename(columns={"L1": "true_L1", "L2": "true_L2"})
    val_pred_df["fold"] = fold
    val_pred_df["split"] = "val"
    val_pred_df["y_true_id"] = y_val
    val_pred_df["y_pred_id"] = val_y_pred
    val_pred_df["pred_L2"] = [L2_ID2LAB[int(i)] for i in val_y_pred]
    val_pred_df["pred_L1"] = val_pred_df["pred_L2"].map(l2_to_l1)

    test_pred_df = test_df[["id", "orig_id", "source_split", "cv_index", "text", "L1", "L2"]].copy()
    test_pred_df = test_pred_df.rename(columns={"L1": "true_L1", "L2": "true_L2"})
    test_pred_df["fold"] = fold
    test_pred_df["split"] = "test"
    test_pred_df["y_true_id"] = y_test
    test_pred_df["y_pred_id"] = test_y_pred
    test_pred_df["pred_L2"] = [L2_ID2LAB[int(i)] for i in test_y_pred]
    test_pred_df["pred_L1"] = test_pred_df["pred_L2"].map(l2_to_l1)

    val_pred_df.to_csv(os.path.join(run_dir, "flat_val_predictions.csv"), index=False)
    test_pred_df.to_csv(os.path.join(run_dir, "flat_test_predictions.csv"), index=False)

    cm_val = confusion_matrix(val_pred_df["true_L2"], val_pred_df["pred_L2"], labels=ALL_L2_LABELS)
    cm_test = confusion_matrix(test_pred_df["true_L2"], test_pred_df["pred_L2"], labels=ALL_L2_LABELS)
    plot_confusion(cm_val, ALL_L2_LABELS, f"Fold {fold} VAL L2 confusion", os.path.join(run_dir, "cm_val_L2.png"))
    plot_confusion(cm_test, ALL_L2_LABELS, f"Fold {fold} TEST L2 confusion", os.path.join(run_dir, "cm_test_L2.png"))

    cm_parent_test = confusion_matrix(test_pred_df["true_L1"], test_pred_df["pred_L1"], labels=ALL_L1_LABELS)
    plot_confusion(cm_parent_test, ALL_L1_LABELS, f"Fold {fold} TEST parent confusion", os.path.join(run_dir, "cm_test_parent.png"), annotate_threshold=20)

    val_flat = flat_metrics_from_labels(val_pred_df["true_L2"].tolist(), val_pred_df["pred_L2"].tolist())
    test_flat = flat_metrics_from_labels(test_pred_df["true_L2"].tolist(), test_pred_df["pred_L2"].tolist())
    val_h = hierarchical_path_metrics(val_pred_df)
    test_h = hierarchical_path_metrics(test_pred_df)
    test_err = error_decomposition(test_pred_df)

    best_ckpt = getattr(trainer.state, "best_model_checkpoint", None)
    best_metric = getattr(trainer.state, "best_metric", None)

    row = {
        "run_id": run_id,
        "stage": "flat_L2_5fold_cv",
        "fold": fold,
        "val_fold": int(split_spec["val_fold"]),
        "train_size": int(len(train_df)),
        "val_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "model_key": model_key,
        "model_name": model_name,
        "finetune_mode": finetune_mode,
        "batch_size": int(batch_size),
        "lr": hp["lr"],
        "wd": hp["wd"],
        "warmup_ratio": hp.get("warmup_ratio", 0.06),
        "warmup_steps": int(warmup_steps),
        "max_len": int(MAX_LEN),
        "seed": int(seed),
        "loss_variant": loss_variant,
        "use_focal_loss": bool(loss_variant in {"focal", "weighted_focal"}),
        "focal_gamma": float(FOCAL_GAMMA) if loss_variant in {"focal", "weighted_focal"} else 0.0,
        "use_class_weights": bool(loss_variant in {"weighted_ce", "weighted_focal"}),
        "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
        "output_dir": run_dir,
        "best_model_checkpoint": best_ckpt,
        "best_metric_val_macro_f1": best_metric,
        "loss_curve_saved": bool(loss_saved),
        **prefix_metrics("val", val_flat),
        **prefix_metrics("val", val_h),
        **prefix_metrics("test", test_flat),
        **prefix_metrics("test", test_h),
        **prefix_metrics("test_error", test_err),
        "status": "OK",
    }

    append_master_row(row)

    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, val_pred_df, test_pred_df


In [ ]:

# =========================
# 12) Run all five folds
# =========================
def run_cv_experiment():
    all_rows = []
    all_val_preds = []
    all_test_preds = []

    for model_key, model_name in MODEL_ZOO.items():
        for bs in BATCH_SIZES:
            for ft_mode in FINETUNE_MODES:
                for loss_variant in LOSS_VARIANTS:
                    for split_spec in cv_splits:
                        try:
                            row, val_pred_df, test_pred_df = run_one_cv_fold(
                                full_df=full_df,
                                split_spec=split_spec,
                                model_name=model_name,
                                model_key=model_key,
                                batch_size=bs,
                                finetune_mode=ft_mode,
                                loss_variant=loss_variant,
                                seed=SEED,
                            )
                        except Exception as e:
                            print("FAILED:", model_key, bs, ft_mode, loss_variant, "fold", split_spec["fold"], repr(e))
                            row = {
                                "run_id": f"flat_cv{split_spec['fold']}_{model_key}_{ft_mode}_bs{bs}_loss{loss_variant}_seed{SEED}",
                                "stage": "flat_L2_5fold_cv",
                                "fold": int(split_spec["fold"]),
                                "val_fold": int(split_spec["val_fold"]),
                                "model_key": model_key,
                                "model_name": model_name,
                                "finetune_mode": ft_mode,
                                "batch_size": int(bs),
                                "policy_name": POLICY_NAME,
                                "dropout": float(DROPOUT),
                                "classifier_dropout": float(CLASSIFIER_DROPOUT),
                                "weight_clip": float(MAX_CLASS_WEIGHT_CLIP),
                                "loss_variant": loss_variant,
                                "seed": int(SEED),
                                "status": "FAILED",
                                "error": repr(e),

                            }
                            append_master_row(row)
                            val_pred_df = pd.DataFrame()
                            test_pred_df = pd.DataFrame()

                        all_rows.append(row)
                        if len(val_pred_df):
                            all_val_preds.append(val_pred_df)
                        if len(test_pred_df):
                            all_test_preds.append(test_pred_df)

    results_df = pd.DataFrame(all_rows)
    val_oof = pd.concat(all_val_preds, ignore_index=True) if all_val_preds else pd.DataFrame()
    test_oof = pd.concat(all_test_preds, ignore_index=True) if all_test_preds else pd.DataFrame()

    results_df.to_csv(os.path.join(OUT_DIR, "flat_cv_fold_results.csv"), index=False)
    results_df.to_excel(os.path.join(OUT_DIR, "flat_cv_fold_results.xlsx"), index=False)
    val_oof.to_csv(os.path.join(OUT_DIR, "flat_cv_val_predictions_all_folds.csv"), index=False)
    test_oof.to_csv(os.path.join(OUT_DIR, "flat_cv_test_oof_predictions.csv"), index=False)

    plot_all_loss_overlays(OUT_DIR, out_png="ALL_FOLD_VAL_LOSS_CURVES.png")
    return results_df, val_oof, test_oof


results_df, val_oof, test_oof = run_cv_experiment()
print("Saved fold results and out-of-fold predictions in:", OUT_DIR)
display(results_df)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MASTER updated: /content/drive/MyDrive/Colab Notebooks/PhDThesis-Anno-June/Model Building/Flat_Output/Flat_5Fold_CV_Seed42_Dropout_DeBerta_RefinedFS/MASTER_FLAT_5FOLD_CV_SEED42.csv
Saved fold results and out-of-fold predictions in: /content/drive/MyDrive/Colab Notebooks/PhDThesis-Anno-June/Model Building/Flat_Output/Flat_5Fold_CV_Seed42_Dropout_DeBerta_RefinedFS


,run_id,stage,fold,val_fold,train_size,val_size,test_size,model_key,model_name,finetune_mode,...,test_HF1,test_parent_accuracy,test_error_n_samples,test_error_Correct L2,test_error_Wrong L2 but correct L1,test_error_Wrong L1,test_error_Correct L2 pct,test_error_Wrong L2 but correct L1 pct,test_error_Wrong L1 pct,status
0,flat_cv0_roberta_full_bs8_lossweighted_focal_s...,flat_L2_5fold_cv,0,1,1075,359,359,roberta,roberta-base,full,...,0.809192,0.896936,359,259,63,37,0.721448,0.175487,0.103064,OK
1,flat_cv1_roberta_full_bs8_lossweighted_focal_s...,flat_L2_5fold_cv,1,2,1075,359,359,roberta,roberta-base,full,...,0.825905,0.894150,359,272,49,38,0.757660,0.136490,0.105850,OK
2,flat_cv2_roberta_full_bs8_lossweighted_focal_s...,flat_L2_5fold_cv,2,3,1076,358,359,roberta,roberta-base,full,...,0.805014,0.877437,359,263,52,44,0.732591,0.144847,0.122563,OK
3,flat_cv3_roberta_full_bs8_lossweighted_focal_s...,flat_L2_5fold_cv,3,4,1077,358,358,roberta,roberta-base,full,...,0.817039,0.871508,358,273,39,46,0.762570,0.108939,0.128492,OK
4,flat_cv4_roberta_full_bs8_lossweighted_focal_s...,flat_L2_5fold_cv,4,0,1076,359,358,roberta,roberta-base,full,...,0.833799,0.902235,358,274,49,35,0.765363,0.136872,0.097765,OK


In [ ]:

# =========================
# 13) Aggregate CV report: overall metrics, HF1/HR/HP, error analysis
# =========================
def summarize_cv_fold_results(results_df: pd.DataFrame) -> pd.DataFrame:
    df = results_df.copy()
    if "status" in df.columns:
        df = df[df["status"].eq("OK")].copy()

    group_cols = ["model_key", "model_name", "finetune_mode", "batch_size", "loss_variant", "seed"]
    metric_cols = [
        c for c in df.columns
        if c.startswith("val_") or c.startswith("test_") or c.startswith("test_error_")
    ]
    metric_cols = [c for c in metric_cols if pd.api.types.is_numeric_dtype(df[c])]

    agg_dict = {"n_folds": ("fold", "nunique")}
    for c in metric_cols:
        agg_dict[f"{c}_mean"] = (c, "mean")
        agg_dict[f"{c}_std"] = (c, "std")

    summary = df.groupby(group_cols, dropna=False).agg(**agg_dict).reset_index()
    for c in metric_cols:
        mean_col = f"{c}_mean"
        std_col = f"{c}_std"
        summary[f"{c}_mean±std"] = summary.apply(
            lambda r: f"{r[mean_col]:.4f} ± {0.0 if pd.isna(r[std_col]) else r[std_col]:.4f}",
            axis=1,
        )
    sort_col = "test_macro_f1_mean" if "test_macro_f1_mean" in summary.columns else None
    if sort_col:
        summary = summary.sort_values(sort_col, ascending=False).reset_index(drop=True)
    return summary


cv_summary = summarize_cv_fold_results(results_df)
cv_summary.to_csv(os.path.join(OUT_DIR, "flat_cv_summary_mean_std_over_folds.csv"), index=False)
cv_summary.to_excel(os.path.join(OUT_DIR, "flat_cv_summary_mean_std_over_folds.xlsx"), index=False)

# True out-of-fold performance across all five held-out test folds.
oof_flat = flat_metrics_from_labels(test_oof["true_L2"].tolist(), test_oof["pred_L2"].tolist(), labels=ALL_L2_LABELS)
oof_h = hierarchical_path_metrics(test_oof)
oof_err = error_decomposition(test_oof)
oof_parent = flat_metrics_from_labels(test_oof["true_L1"].tolist(), test_oof["pred_L1"].tolist(), labels=ALL_L1_LABELS)

oof_summary = {
    **prefix_metrics("oof_test", oof_flat),
    **prefix_metrics("oof_test", oof_h),
    **prefix_metrics("oof_parent", oof_parent),
    **prefix_metrics("oof_error", oof_err),
}
oof_summary_df = pd.DataFrame([oof_summary])
oof_summary_df.to_csv(os.path.join(OUT_DIR, "flat_cv_oof_overall_summary.csv"), index=False)
oof_summary_df.to_excel(os.path.join(OUT_DIR, "flat_cv_oof_overall_summary.xlsx"), index=False)

print("Fold mean ± std summary:")
display(cv_summary)
print("OOF test summary over all five held-out test folds:")
display(oof_summary_df)


Fold mean ± std summary:


,model_key,model_name,finetune_mode,batch_size,loss_variant,seed,n_folds,val_fold_mean,val_fold_std,val_size_mean,...,test_HR_mean±std,test_HF1_mean±std,test_parent_accuracy_mean±std,test_error_n_samples_mean±std,test_error_Correct L2_mean±std,test_error_Wrong L2 but correct L1_mean±std,test_error_Wrong L1_mean±std,test_error_Correct L2 pct_mean±std,test_error_Wrong L2 but correct L1 pct_mean±std,test_error_Wrong L1 pct_mean±std
0,roberta,roberta-base,full,8,weighted_focal,42,5,2.0,1.581139,358.6,...,0.8182 ± 0.0118,0.8182 ± 0.0118,0.8885 ± 0.0133,358.6000 ± 0.5477,268.2000 ± 6.7602,50.4000 ± 8.5907,40.0000 ± 4.7434,0.7479 ± 0.0197,0.1405 ± 0.0238,0.1115 ± 0.0133


OOF test summary over all five held-out test folds:


,oof_test_macro_f1,oof_test_weighted_f1,oof_test_accuracy,oof_test_balanced_acc,oof_test_precision_macro,oof_test_recall_macro,oof_test_HP,oof_test_HR,oof_test_HF1,oof_test_parent_accuracy,...,oof_parent_balanced_acc,oof_parent_precision_macro,oof_parent_recall_macro,oof_error_n_samples,oof_error_Correct L2,oof_error_Wrong L2 but correct L1,oof_error_Wrong L1,oof_error_Correct L2 pct,oof_error_Wrong L2 but correct L1 pct,oof_error_Wrong L1 pct
0,0.69085,0.746977,0.747909,0.692704,0.701424,0.692704,0.818182,0.818182,0.818182,0.888455,...,0.841692,0.851253,0.841692,1793,1341,252,200,0.747909,0.140547,0.111545


In [ ]:

# =========================
# 14) Confusion matrices: aggregate OOF L2 and parent confusion
# =========================
cm_l2_oof = confusion_matrix(test_oof["true_L2"], test_oof["pred_L2"], labels=ALL_L2_LABELS)
cm_parent_oof = confusion_matrix(test_oof["true_L1"], test_oof["pred_L1"], labels=ALL_L1_LABELS)

pd.DataFrame(cm_l2_oof, index=ALL_L2_LABELS, columns=ALL_L2_LABELS).to_csv(os.path.join(OUT_DIR, "cm_oof_L2.csv"))
pd.DataFrame(cm_parent_oof, index=ALL_L1_LABELS, columns=ALL_L1_LABELS).to_csv(os.path.join(OUT_DIR, "cm_oof_parent.csv"))

plot_confusion(cm_l2_oof, ALL_L2_LABELS, "OOF TEST L2 confusion matrix", os.path.join(OUT_DIR, "cm_oof_L2.png"))
plot_confusion(cm_parent_oof, ALL_L1_LABELS, "OOF TEST parent confusion matrix", os.path.join(OUT_DIR, "cm_oof_parent.png"), annotate_threshold=20)

print("Saved aggregate confusion matrices:")
print(" -", os.path.join(OUT_DIR, "cm_oof_L2.png"))
print(" -", os.path.join(OUT_DIR, "cm_oof_parent.png"))


Saved aggregate confusion matrices:
 - /content/drive/MyDrive/Colab Notebooks/PhDThesis-Anno-June/Model Building/Flat_Output/Flat_5Fold_CV_Seed42_Dropout_DeBerta_RefinedFS/cm_oof_L2.png
 - /content/drive/MyDrive/Colab Notebooks/PhDThesis-Anno-June/Model Building/Flat_Output/Flat_5Fold_CV_Seed42_Dropout_DeBerta_RefinedFS/cm_oof_parent.png


In [ ]:

# =========================
# 15) Per-class analysis over folds and OOF test set
# =========================
per_class_rows = []
for fold, sub in test_oof.groupby("fold"):
    rep = classification_report_df(sub["true_L2"].tolist(), sub["pred_L2"].tolist(), ALL_L2_LABELS)
    rep["fold"] = int(fold)
    rep["L1"] = rep["label"].map(l2_to_l1)
    per_class_rows.append(rep)
per_class_by_fold = pd.concat(per_class_rows, ignore_index=True)
per_class_by_fold = per_class_by_fold.rename(columns={"label": "L2"})

per_class_summary = (
    per_class_by_fold
    .groupby(["L1", "L2"], dropna=False)
    .agg(
        n_test_folds_with_support=("support", lambda s: int((s > 0).sum())),
        total_support=("support", "sum"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .reset_index()
    .sort_values(["L1", "f1_mean", "total_support"], ascending=[True, True, False])
)

# OOF per-class report; this is usually cleaner for the paper because every sample is counted once.
per_class_oof = classification_report_df(test_oof["true_L2"].tolist(), test_oof["pred_L2"].tolist(), ALL_L2_LABELS)
per_class_oof = per_class_oof.rename(columns={"label": "L2"})
per_class_oof["L1"] = per_class_oof["L2"].map(l2_to_l1)
per_class_oof = per_class_oof[["L1", "L2", "support", "precision", "recall", "f1"]].sort_values(["L1", "f1", "support"], ascending=[True, True, False])

per_class_by_fold.to_csv(os.path.join(OUT_DIR, "per_class_by_fold.csv"), index=False)
per_class_summary.to_csv(os.path.join(OUT_DIR, "per_class_summary_mean_std_over_folds.csv"), index=False)
per_class_oof.to_csv(os.path.join(OUT_DIR, "per_class_oof_report.csv"), index=False)

per_class_summary.to_excel(os.path.join(OUT_DIR, "per_class_summary_mean_std_over_folds.xlsx"), index=False)
per_class_oof.to_excel(os.path.join(OUT_DIR, "per_class_oof_report.xlsx"), index=False)

print("Worst OOF L2 classes by F1:")
display(per_class_oof.head(15))


Worst OOF L2 classes by F1:


,L1,L2,support,precision,recall,f1
11,compatibility,interoperability,97.0,0.743902,0.628866,0.681564
15,maintainability,modularity,23.0,0.565217,0.565217,0.565217
14,maintainability,modifiability,66.0,0.711864,0.636364,0.672000
20,maintainability,testability,27.0,0.724138,0.777778,0.750000
18,performanceefficiency,resourceutilization,58.0,0.632353,0.741379,0.682540
6,performanceefficiency,capacity,80.0,0.835443,0.825000,0.830189
21,performanceefficiency,timebehaviour,185.0,0.820896,0.891892,0.854922
19,performanceefficiency,scalability,30.0,0.925926,0.833333,0.877193
9,portability,installability,19.0,0.500000,0.210526,0.296296
2,portability,adaptability,109.0,0.715385,0.853211,0.778243


In [ ]:

# =========================
# 16) Per-parent analysis over folds and OOF test set
# =========================
per_parent_rows = []
for fold, sub in test_oof.groupby("fold"):
    parent_rep = classification_report_df(sub["true_L1"].tolist(), sub["pred_L1"].tolist(), ALL_L1_LABELS)
    parent_rep = parent_rep.rename(columns={"label": "parent"})
    parent_rep["fold"] = int(fold)
    within = per_parent_within_l2_metrics(sub, l2_to_l1, ALL_L1_LABELS)
    merged = parent_rep.merge(within, on="parent", how="left", suffixes=("_parent_cls", ""))
    per_parent_rows.append(merged)

per_parent_by_fold = pd.concat(per_parent_rows, ignore_index=True)
per_parent_summary = (
    per_parent_by_fold
    .groupby("parent", dropna=False)
    .agg(
        n_test_folds_with_support=("support_x", lambda s: int((s > 0).sum()) if "support_x" in per_parent_by_fold.columns else int((s > 0).sum())),
    ) if False else None
)

# Rebuild summary explicitly to avoid suffix ambiguity across sklearn versions.
support_col = "support" if "support" in per_parent_by_fold.columns else "support_x"
summary_cols = [
    "precision", "recall", "f1", "parent_recall_as_parent",
    "within_parent_l2_macro_f1", "within_parent_l2_weighted_f1",
    "sibling_error_rate", "cross_parent_error_rate",
]
agg = {
    "n_test_folds_with_support": (support_col, lambda s: int((s > 0).sum())),
    "total_support": (support_col, "sum"),
}
for c in summary_cols:
    if c in per_parent_by_fold.columns:
        agg[f"{c}_mean"] = (c, "mean")
        agg[f"{c}_std"] = (c, "std")
per_parent_summary = per_parent_by_fold.groupby("parent", dropna=False).agg(**agg).reset_index()

per_parent_oof_cls = classification_report_df(test_oof["true_L1"].tolist(), test_oof["pred_L1"].tolist(), ALL_L1_LABELS)
per_parent_oof_cls = per_parent_oof_cls.rename(columns={"label": "parent"})
per_parent_oof_within = per_parent_within_l2_metrics(test_oof, l2_to_l1, ALL_L1_LABELS)
per_parent_oof = per_parent_oof_cls.merge(per_parent_oof_within, on="parent", how="left")

per_parent_by_fold.to_csv(os.path.join(OUT_DIR, "per_parent_by_fold.csv"), index=False)
per_parent_summary.to_csv(os.path.join(OUT_DIR, "per_parent_summary_mean_std_over_folds.csv"), index=False)
per_parent_oof.to_csv(os.path.join(OUT_DIR, "per_parent_oof_report.csv"), index=False)
per_parent_summary.to_excel(os.path.join(OUT_DIR, "per_parent_summary_mean_std_over_folds.xlsx"), index=False)
per_parent_oof.to_excel(os.path.join(OUT_DIR, "per_parent_oof_report.xlsx"), index=False)

print("OOF per-parent report:")
display(per_parent_oof.sort_values("within_parent_l2_macro_f1"))


OOF per-parent report:


,parent,precision,recall,f1,support_x,support_y,parent_recall_as_parent,within_parent_l2_macro_f1,within_parent_l2_weighted_f1,sibling_error_rate,cross_parent_error_rate
3,portability,0.775362,0.835938,0.804511,128.0,128,0.835938,0.598822,0.803523,0.078125,0.164062
6,usability,0.913947,0.870056,0.891462,354.0,354,0.870056,0.677316,0.720913,0.197740,0.129944
5,security,0.934211,0.946667,0.940397,450.0,450,0.946667,0.707325,0.767233,0.202222,0.053333
1,maintainability,0.783784,0.750000,0.766520,116.0,116,0.750000,0.744884,0.748191,0.094828,0.250000
0,compatibility,0.743902,0.628866,0.681564,97.0,97,0.628866,0.772152,0.772152,0.000000,0.371134
4,reliability,0.911565,0.908475,0.910017,295.0,295,0.908475,0.803233,0.831566,0.111864,0.091525
2,performanceefficiency,0.896000,0.951841,0.923077,353.0,353,0.951841,0.846618,0.868116,0.104816,0.048159


In [ ]:

# =========================
# 17) Error analysis over folds and by parent/class
# =========================
error_rows = []
for fold, sub in test_oof.groupby("fold"):
    row = {"fold": int(fold)}
    row.update(error_decomposition(sub))
    error_rows.append(row)
error_by_fold = pd.DataFrame(error_rows)

error_summary = pd.DataFrame([{
    "n_folds": int(error_by_fold["fold"].nunique()),
    **{f"{c}_mean": error_by_fold[c].mean() for c in error_by_fold.columns if c != "fold"},
    **{f"{c}_std": error_by_fold[c].std() for c in error_by_fold.columns if c != "fold"},
}])

# Error type for every OOF prediction.
err_df = test_oof.copy()
correct_l2 = err_df["true_L2"].eq(err_df["pred_L2"])
correct_parent = err_df["true_L1"].eq(err_df["pred_L1"])
err_df["error_type"] = np.select(
    [correct_l2, (~correct_l2) & correct_parent, ~correct_parent],
    ["Correct L2", "Wrong L2 but correct L1", "Wrong L1"],
    default="Unknown",
)

error_by_parent = (
    err_df.groupby(["true_L1", "error_type"]).size()
    .reset_index(name="n")
    .merge(err_df.groupby("true_L1").size().reset_index(name="parent_support"), on="true_L1")
)
error_by_parent["pct_within_parent"] = error_by_parent["n"] / error_by_parent["parent_support"]

error_by_class = (
    err_df.groupby(["true_L1", "true_L2", "error_type"]).size()
    .reset_index(name="n")
    .merge(err_df.groupby(["true_L1", "true_L2"]).size().reset_index(name="class_support"), on=["true_L1", "true_L2"])
)
error_by_class["pct_within_class"] = error_by_class["n"] / error_by_class["class_support"]

error_by_fold.to_csv(os.path.join(OUT_DIR, "error_analysis_by_fold.csv"), index=False)
error_summary.to_csv(os.path.join(OUT_DIR, "error_analysis_summary_mean_std.csv"), index=False)
err_df.to_csv(os.path.join(OUT_DIR, "oof_predictions_with_error_type.csv"), index=False)
error_by_parent.to_csv(os.path.join(OUT_DIR, "error_analysis_by_parent.csv"), index=False)
error_by_class.to_csv(os.path.join(OUT_DIR, "error_analysis_by_class.csv"), index=False)

print("Error decomposition by fold:")
display(error_by_fold)
print("Error decomposition by parent:")
display(error_by_parent.pivot_table(index="true_L1", columns="error_type", values="pct_within_parent", fill_value=0))


Error decomposition by fold:


,fold,n_samples,Correct L2,Wrong L2 but correct L1,Wrong L1,Correct L2 pct,Wrong L2 but correct L1 pct,Wrong L1 pct
0,0,359,259,63,37,0.721448,0.175487,0.103064
1,1,359,272,49,38,0.757660,0.136490,0.105850
2,2,359,263,52,44,0.732591,0.144847,0.122563
3,3,358,273,39,46,0.762570,0.108939,0.128492
4,4,358,274,49,35,0.765363,0.136872,0.097765


Error decomposition by parent:


error_type,Correct L2,Wrong L1,Wrong L2 but correct L1
true_L1,,,
compatibility,0.628866,0.371134,0.000000
maintainability,0.655172,0.250000,0.094828
performanceefficiency,0.847025,0.048159,0.104816
portability,0.757812,0.164062,0.078125
reliability,0.796610,0.091525,0.111864
security,0.744444,0.053333,0.202222
usability,0.672316,0.129944,0.197740


## What to report in the paper

Use the following files as the core reporting artifacts:

1. `flat_cv_oof_overall_summary.xlsx` — main held-out CV estimate using all out-of-fold test predictions.
2. `flat_cv_summary_mean_std_over_folds.xlsx` — mean ± standard deviation across the five test folds.
3. `per_class_oof_report.xlsx` and `per_class_summary_mean_std_over_folds.xlsx` — L2 class analysis.
4. `per_parent_oof_report.xlsx` and `per_parent_summary_mean_std_over_folds.xlsx` — L1/parent analysis.
5. `error_analysis_by_fold.csv`, `error_analysis_by_parent.csv`, and `error_analysis_by_class.csv` — hierarchical error decomposition.
6. `cm_oof_L2.png`, `cm_oof_parent.png`, and `ALL_FOLD_VAL_LOSS_CURVES.png` — figures for the result section.
